In [2]:
import math
import torch
import torch.nn as nn

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.dim = dim
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
    
    def forward(self, x):
        # X shape is (batch, seq_len, dim)
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        attention_weight = torch.softmax(torch.matmul(
            q, k.transpose(-1, -2)
        ) / math.sqrt(self.dim),
        dim = -1
        )
        output = torch.matmul(attention_weight, v)
        return output

x = torch.rand(3, 4, 5)
SA = SelfAttention(dim = 5)
output = SA(x)
output


tensor([[[ 0.1799,  0.2996,  0.0781, -0.2534,  1.0708],
         [ 0.1796,  0.2999,  0.0776, -0.2529,  1.0705],
         [ 0.1793,  0.2995,  0.0780, -0.2528,  1.0712],
         [ 0.1804,  0.3002,  0.0779, -0.2544,  1.0702]],

        [[ 0.1801,  0.3232,  0.0960, -0.2692,  1.0922],
         [ 0.1846,  0.3203,  0.0917, -0.2627,  1.0842],
         [ 0.1844,  0.3226,  0.0951, -0.2702,  1.0880],
         [ 0.1903,  0.3212,  0.0931, -0.2698,  1.0815]],

        [[ 0.1614,  0.2072, -0.0710, -0.0087,  0.8954],
         [ 0.1608,  0.2076, -0.0701, -0.0091,  0.8972],
         [ 0.1617,  0.2070, -0.0710, -0.0087,  0.8951],
         [ 0.1614,  0.2073, -0.0710, -0.0086,  0.8953]]],
       grad_fn=<UnsafeViewBackward0>)

In [8]:
class SelfAttentionfull(nn.Module):
    def __init__(self, dim: int=512, rate: float=0.1):
        super().__init__()
        
        self.dim = dim
        self.proj = nn.Linear(dim, dim * 3)
        self.o_proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(rate)

    def forward(self, x, attention_mask=None):
        qkv = self.proj(x)  # (batch, seq_len, dim*3)
        q, k, v = torch.split(
            qkv, self.dim, dim=-1
        )
        attention_value = q @ k.transpose(-1, -2) / math.sqrt(self.dim)  #(batch, seq_len, seq_len)
        if attention_mask is not None:
            attention_value = attention_value.masked_fill(
                attention_mask==0,
                float('-inf')
            )
        attention_weight = torch.softmax(attention_value, dim=-1)
        attention_weight = self.dropout(attention_weight)
        result = attention_weight @ v
        output = self.o_proj(result)
        return output
        

In [12]:
x = torch.rand(3, 4, 5)
attention_mask = torch.tensor([
    [1, 1, 1, 0],
    [1, 1, 0, 0],
    [1, 0, 0, 0]
])  # (3, 4)  目标是(3, 4, 4)
attention_mask = attention_mask.unsqueeze(dim=1) #(3, 1, 4)
# print(attention_mask.size())
attention_mask = attention_mask.repeat(1, 4, 1)
# print(attention_mask.size())
SA2 = SelfAttentionfull(dim=x.size(-1))
output = SA2(x, attention_mask=attention_mask)
output

tensor([[[ 0.1875,  0.2865,  0.0601,  0.4289, -0.0789],
         [ 0.1948,  0.2889,  0.0598,  0.4298, -0.0803],
         [ 0.1918,  0.2874,  0.0588,  0.4279, -0.0783],
         [ 0.1904,  0.2869,  0.0587,  0.4276, -0.0779]],

        [[ 0.2060,  0.3133, -0.1082,  0.3070, -0.0815],
         [ 0.2070,  0.3132, -0.1082,  0.3073, -0.0816],
         [ 0.2058,  0.3133, -0.1082,  0.3069, -0.0814],
         [ 0.2064,  0.3132, -0.1082,  0.3071, -0.0815]],

        [[ 0.0716,  0.2911, -0.0246,  0.3536, -0.0828],
         [ 0.0716,  0.2911, -0.0246,  0.3536, -0.0828],
         [ 0.0716,  0.2911, -0.0246,  0.3536, -0.0828],
         [-0.1325,  0.3510, -0.0325,  0.4202, -0.2688]]],
       grad_fn=<ViewBackward0>)

In [1]:
import torch
import torch.nn as nn
import math

In [4]:
class SelfAttention(nn.Module):
    def __init__(self, dim: int=512, rate: float=0.1):
        super().__init__()

        self.dim = dim
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.o_proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(rate)
    
    def forward(self, x, attention_mask=None):
        # X shape is (batch, seq_len, dim)
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        attention_value = q @ k.transpose(-1, -2) / math.sqrt(self.dim)
        if attention_mask is not None:
            attention_value = attention_value.masked_fill(
                attention_mask==0,
                float('-inf')
            )
        attention_weight = torch.softmax(
            attention_value,
            dim=-1
        ) # (batch, seq_len, seq_len)
        attention_weight = self.dropout(attention_weight)
        result = attention_weight @ v
        output = self.o_proj(result)
        return output

x = torch.rand(3, 4, 5)
attention_mask = torch.tensor([
    [1, 1, 1, 0],
    [1, 1, 0, 0],
    [1, 0, 0, 0]
])  # (3, 4)
attention_mask = attention_mask.unsqueeze(1).repeat(1, 4, 1)  # (3, 4, 4)
SA = SelfAttention(x.size(-1))
output = SA(x, attention_mask)
output

tensor([[[ 0.7229, -0.1989, -0.4928, -0.0144, -0.3504],
         [ 0.7227, -0.1988, -0.4929, -0.0151, -0.3499],
         [ 0.6583, -0.1666, -0.4126,  0.0287, -0.3520],
         [ 0.6257, -0.1702, -0.3771,  0.0592, -0.3964]],

        [[ 0.4611, -0.1219, -0.2160,  0.0512, -0.2810],
         [ 0.5061, -0.1774, -0.2776, -0.0244, -0.2062],
         [ 0.4623, -0.1228, -0.2174,  0.0493, -0.2771],
         [ 0.5060, -0.1769, -0.2773, -0.0241, -0.2053]],

        [[ 0.7531, -0.2229, -0.5154, -0.1172, -0.1400],
         [ 0.7531, -0.2229, -0.5154, -0.1172, -0.1400],
         [ 0.7531, -0.2229, -0.5154, -0.1172, -0.1400],
         [ 0.7531, -0.2229, -0.5154, -0.1172, -0.1400]]],
       grad_fn=<ViewBackward0>)

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, dim: int=512, head_num: int=8, rate: float=0.1):
        super().__init__()

        self.dim = dim
        self.head_num = head_num
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.o_proj = nn.Linear(dim, dim)
        self.head_dim = dim // head_num
        self.dropout = nn.Dropout(rate)

    def forward(self, x, attention_mask=None):
        # X shape is (batch, seq_len, dim)
        batch, seq_len, dim = x.shape
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(batch, seq_len, self.head_num, self.head_dim).transpose(1, 2)  # (batch, head_num, seq_len, head_dim)
        k = k.view(batch, seq_len, self.head_num, self.head_dim).transpose(1, 2)
        v = v.view(batch, seq_len, self.head_num, self.head_dim).transpose(1, 2)
        attention_value = q @ k.transpose(-1, -2) / math.sqrt(self.head_dim)  #(batch, head_num, seq_len, seq_len)
        if attention_mask is not None:
            attention_value = attention_value.masked_fill(
                attention_mask==0,
                float('-inf')
            )
        attention_weight = torch.softmax(attention_value, dim=-1)
        attention_weight = self.dropout(attention_weight)
        result = attention_weight @ v  # (batch, head_num, seq_len, head_dim)
        result = result.transpose(1, 2).contiguous()
        result = result.view(batch, seq_len, -1)
        output = self.o_proj(result)

        return output


In [9]:
x = torch.rand(3, 4, 128)
attention_mask = torch.tensor([
    [1, 1, 1, 0],
    [1, 1, 0, 0],
    [1, 0, 0, 0]
]).unsqueeze(1).repeat(1, 4, 1).unsqueeze(1).repeat(1, 8, 1, 1)
print(attention_mask.shape)
MHA = MultiHeadAttention(x.size(-1), head_num=8)
output = MHA(x, attention_mask)
output

torch.Size([3, 8, 4, 4])


tensor([[[ 0.2353, -0.1475, -0.0693,  ...,  0.1030, -0.4143, -0.0126],
         [ 0.1361, -0.0966, -0.0678,  ...,  0.0537, -0.3337,  0.0454],
         [ 0.1900, -0.1393, -0.1042,  ...,  0.0988, -0.3588, -0.0296],
         [ 0.1415, -0.0934, -0.0261,  ...,  0.1022, -0.3373,  0.0065]],

        [[ 0.1404, -0.1824, -0.1349,  ...,  0.1377, -0.4709, -0.0187],
         [ 0.1462, -0.0862, -0.1479,  ...,  0.0148, -0.4550,  0.0538],
         [ 0.1401, -0.1633, -0.1676,  ...,  0.1381, -0.5295,  0.0231],
         [ 0.0961, -0.0961, -0.1215,  ...,  0.0476, -0.4589, -0.0165]],

        [[ 0.0822, -0.1769, -0.2052,  ...,  0.1901, -0.2776,  0.1439],
         [ 0.1762, -0.1734, -0.2284,  ...,  0.1574, -0.3170,  0.0349],
         [ 0.1948, -0.1353, -0.1586,  ...,  0.0609, -0.2905,  0.0908],
         [ 0.1997, -0.0638, -0.0174,  ...,  0.0307, -0.2905, -0.0096]]],
       grad_fn=<ViewBackward0>)

In [1]:
import torch
import torch.nn as nn
import math
from dataclasses import dataclass

In [1]:
@dataclass
class GQAConfig:
    dim: int
    q_heads: int
    kv_heads: int
    dropout: float

class GroupQueryAttention(nn.Module):
    def __init__(self, config: GQAConfig):
        super().__init__()
        self.config = config
        self.q_heads = config.q_heads
        self.kv_heads = config.kv_heads
        self.dim = config.dim
        self.head_dim = config.dim // config.q_heads
        self.q_proj = nn.Linear(config.dim, config.q_heads * self.head_dim)
        self.k_proj = nn.Linear(config.dim, config.kv_heads * self.head_dim)
        self.v_proj = nn.Linear(config.dim, config.kv_heads * self.head_dim)
        self.o_proj = nn.Linear(config.dim, config.dim)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, attention_mask=None):
        batch, seq_len, _ = x.shape
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(batch, seq_len, self.q_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch, seq_len, self.kv_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch, seq_len, self.kv_heads, self.head_dim).transpose(1, 2)
        
        k = k.repeat_interleave(self.q_heads // self.kv_heads, dim=1)
        v = v.repeat_interleave(self.q_heads // self.kv_heads, dim=1)

        attention_value = q @ k.transpose(-1, -2) / math.sqrt(self.head_dim)
        if attention_mask is not None:
            attention_value = attention_value.masked_fill(
                attention_mask==0,
                float('-inf')
            )
        attention_weight = torch.softmax(attention_value, dim=-1)
        attention_weight = self.dropout(attention_weight)
        output = attention_weight @ v
        output = output.transpose(1, 2).contiguous()
        output = output.view(batch, seq_len, -1)
        
        return self.o_proj(output)

NameError: name 'dataclass' is not defined

In [2]:
def rrf(rankings, k: int=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank)
    
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

In [12]:
def rrf(lists, k: int=60):
    scores = {}
    for list in lists:
        for i, id in enumerate(list):
            scores[id] = scores.get(id, 0) + 1.0 / (i + k)
    return scores

lists = [
    ['A', 'B', 'C', 'D'],
    ['C', "B", "A", 'D']
]

scores = rrf(lists)
sorted(scores.items(), key=lambda x: x[1], reverse=True)

[('A', 0.03279569892473118),
 ('C', 0.03279569892473118),
 ('B', 0.03278688524590164),
 ('D', 0.031746031746031744)]